# SPARQL Inferencing — User Guide

`entailment` is a parameter of `.query()`, not a setting on the graph — the same `StarLayerGraph` can be queried with different entailment on different calls:

- **`entailment="rdfs"`** — rewrites the query at query time for the full RDFS ruleset's "data" rules: `rdf:type`/`rdfs:subClassOf`, `rdfs:domain`/`rdfs:range`, and `rdfs:subPropertyOf` entailment. No data is copied.
- **`entailment="owl-rl"`** — for what `"rdfs"` rewriting can't express (genuine OWL constructs like `owl:equivalentClass`), the graph is queried through a materialized RDFS/OWL-RL closure via [`infer()`](02b-graphs-inferencing.ipynb) instead — cached on the graph and reused across calls until something actually changes (tunable via the separate `infer=` parameter, section 3).

The [SPARQL 1.1/1.2 Entailment Regimes](https://www.w3.org/TR/sparql12-entailment/) spec defines six regimes. Here's what this project covers, and what it doesn't:

| Regime | Status |
|---|---|
| Simple / RDF-1.2-aware matching | ✅ Done — the default, no `entailment=` needed |
| RDFS Entailment | ✅ Done — `entailment="rdfs"` (section 1), the full ruleset's "data" rules via query-time rewrite; deliberately excludes the "vocabulary-level" rules (universal `rdfs:Resource` typing, the fixed axiomatic triples) - see `starsparql.entailment_rdfs`'s module docstring for the exact rule-by-rule scope and why |
| OWL 2 RL (a decidable rule-based fragment of OWL 2 RDF-Based Semantics) | ✅ Done — `entailment="owl-rl"` (section 3), covers `owl:equivalentClass`, `owl:TransitiveProperty`, and the rest of the OWL 2 RL profile |
| OWL 2 RDF-Based Semantics (full, beyond the RL fragment) | ❌ Not planned |
| D-Entailment (datatype canonicalization) | ❌ Not planned |
| OWL 2 Direct Semantics | ❌ Not planned — needs a DL reasoner (HermiT-class); tracked separately in the [Inferencing guide](02b-graphs-inferencing.ipynb)'s "Where to go next" |
| RIF Core Entailment | ❌ Not planned — no RIF interpreter in this stack |

Also planned but not started: delegating to a reasoning-enabled backend (e.g. a Fuseki/Jena dataset configured with its own inference model) for whichever regime *it* supports, rather than StarLayerGraph doing the reasoning itself.

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed.

Run cells from top to bottom — later sections may reuse variables from earlier sections.

In [1]:
from starlayergraph import StarLayerGraph, Namespace, RDF

EX = Namespace("http://example.org/")

## 1. `entailment="rdfs"` — query-time RDFS entailment

`ex:alice` is only ever asserted `a ex:Manager`. `ex:Manager rdfs:subClassOf ex:Employee` means `ex:alice a ex:Employee` is entailed, but never actually written as a triple. A plain query is simple entailment only — it misses this. `entailment="rdfs"` rewrites the query's `rdf:type` patterns (via an `rdflib` property path, `rdf:type/rdfs:subClassOf*`) so the match happens at query time, against the graph's current data, without touching it. (Section 1.b covers the rest of the RDFS ruleset this same `entailment="rdfs"` value also handles — `rdfs:domain`/`rdfs:range`/`rdfs:subPropertyOf`.)

In [2]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:Employee rdfs:subClassOf ex:Person .
    ex:alice a ex:Manager .
''', format="turtle12")

QUERY = "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Employee }"

print("plain (no entailment):", [str(r.x) for r in g.query(QUERY)])
print("entailment=\"rdfs\":       ", [str(r.x) for r in g.query(QUERY, entailment="rdfs")])
print("triple count unchanged - no data was copied or added:", len(g))

plain (no entailment): []
entailment="rdfs":        ['http://example.org/alice']
triple count unchanged - no data was copied or added: 3


### 1.a Reflexivity and multi-hop transitivity

`rdfs:subClassOf` entailment is both reflexive (a direct type still matches itself) and transitive across any number of hops. Binding the object as a variable yields every class `ex:alice` is entailed to have — direct and inherited alike.

In [3]:
rows = g.query("PREFIX ex: <http://example.org/> SELECT ?type WHERE { ex:alice a ?type }", entailment="rdfs")
print("every entailed type:", sorted(str(r.type) for r in rows))

rows = g.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Person }", entailment="rdfs")
print("two hops away (Manager -> Employee -> Person):", [str(r.x) for r in rows])

every entailed type: ['http://example.org/Employee', 'http://example.org/Manager', 'http://example.org/Person']
two hops away (Manager -> Employee -> Person): ['http://example.org/alice']


### 1.b `rdfs:domain`, `rdfs:range`, and `rdfs:subPropertyOf`

The same `entailment="rdfs"` value also covers the rest of the RDFS "data" ruleset — a property's declared `rdfs:domain`/`rdfs:range` entails a type for its subject/object, and `rdfs:subPropertyOf` lets a query for a general relation also match its more specific sub-relations. None of this needs a data copy either.

In [4]:
g2 = StarLayerGraph()
g2.bind("ex", EX)
g2.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:worksAt rdfs:domain ex:Person ;
               rdfs:range ex:Organization .
    ex:hasParent rdfs:subPropertyOf ex:hasRelative .
    ex:alice ex:worksAt ex:Acme .
    ex:alice ex:hasParent ex:bob .
''', format="turtle12")

rows = g2.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Person }", entailment="rdfs")
print("alice is a Person (via rdfs:domain):     ", [str(r.x) for r in rows])

rows = g2.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Organization }", entailment="rdfs")
print("Acme is an Organization (via rdfs:range):", [str(r.x) for r in rows])

rows = g2.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ex:alice ex:hasRelative ?x }", entailment="rdfs")
print("alice hasRelative bob (via subPropertyOf):", [str(r.x) for r in rows])

alice is a Person (via rdfs:domain):      ['http://example.org/alice']
Acme is an Organization (via rdfs:range): ['http://example.org/Acme']
alice hasRelative bob (via subPropertyOf): ['http://example.org/bob']


## 2. Always reflects current data

Because `entailment="rdfs"` rewrites the *query*, not the data, a fact added after the last query is entailed correctly on the very next one — there's no separate "re-run reasoning" step to remember. `entailment="owl-rl"` (section 3) has the same property by default too, for a different reason: it recomputes its cached closure automatically whenever the graph actually changes.

In [5]:
print("before adding bob:", [str(r.x) for r in g.query(QUERY, entailment="rdfs")])

g.add((EX.bob, RDF.type, EX.Manager))
print("after adding bob: ", sorted(str(r.x) for r in g.query(QUERY, entailment="rdfs")))

before adding bob: ['http://example.org/alice']
after adding bob:  ['http://example.org/alice', 'http://example.org/bob']


## 3. `entailment="owl-rl"` — query a materialized, cached closure

`entailment="rdfs"` (sections 1-2) covers the full RDFS "data" ruleset via query-time rewrite. It does **not** cover genuine OWL constructs like `owl:equivalentClass` or `owl:TransitiveProperty` — those depend on rule interactions (e.g. congruence closure for `owl:sameAs`) that can need multiple derivation rounds, not expressible as a single bounded rewrite. `entailment="owl-rl"` handles them by materializing a full RDFS/OWL-RL closure (the same [`infer(profile="owl-rl")`](02b-graphs-inferencing.ipynb) from the Inferencing guide) and running the query against it. That closure is cached on the graph and reused across calls — recomputed only when needed (section 3.a controls exactly when via `infer=`).

In [6]:
g_owl = StarLayerGraph()
g_owl.bind("ex", EX)
g_owl.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
''', format="turtle12")

QUERY_OWL = "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:TeamLead }"
print("entailment=\"rdfs\"   (misses owl:equivalentClass):", [str(r.x) for r in g_owl.query(QUERY_OWL, entailment="rdfs")])
print("entailment=\"owl-rl\" (catches it):                ", [str(r.x) for r in g_owl.query(QUERY_OWL, entailment="owl-rl")])

entailment="rdfs"   (misses owl:equivalentClass): []
entailment="owl-rl" (catches it):                 ['http://example.org/alice']


### 3.a Caching: the `infer=` parameter

By default (`infer="changed"`), a query with `entailment="owl-rl"` reuses the cached closure as long as the graph hasn't been edited since it was computed, and recomputes automatically the moment it has — so, like `entailment="rdfs"`, it always reflects current data, without redoing the reasoning work on every single call.

In [7]:
g_owl.add((EX.bob, RDF.type, EX.Manager))
rows = g_owl.query(QUERY_OWL, entailment="owl-rl")
print("bob picked up automatically (default infer='changed'):", sorted(str(r.x) for r in rows))

bob picked up automatically (default infer='changed'): ['http://example.org/alice', 'http://example.org/bob']


`infer="cached"` reuses whatever's cached no matter what — even if the graph changed since, it won't notice. `infer="always"` is the opposite: it ignores the cache entirely and recomputes every time.

In [8]:
g_owl.add((EX.carol, RDF.type, EX.Manager))
rows = g_owl.query(QUERY_OWL, entailment="owl-rl")
print("default (changed), picks up carol:", sorted(str(r.x) for r in rows))

g_owl.add((EX.dave, RDF.type, EX.Manager))
stale = g_owl.query(QUERY_OWL, entailment="owl-rl", infer="cached")
print("infer=\"cached\" (misses dave - stale on purpose):", sorted(str(r.x) for r in stale))

fresh = g_owl.query(QUERY_OWL, entailment="owl-rl", infer="always")
print("infer=\"always\" (picks up dave):                 ", sorted(str(r.x) for r in fresh))

default (changed), picks up carol: ['http://example.org/alice', 'http://example.org/bob', 'http://example.org/carol']
infer="cached" (misses dave - stale on purpose): ['http://example.org/alice', 'http://example.org/bob', 'http://example.org/carol']
infer="always" (picks up dave):                  ['http://example.org/alice', 'http://example.org/bob', 'http://example.org/carol', 'http://example.org/dave']


### 3.b Which one to use

- **`entailment="rdfs"`** — no copy, cheapest per query. The full RDFS "data" ruleset: `rdf:type`/`rdfs:subClassOf`/`rdfs:subPropertyOf`/`rdfs:domain`/`rdfs:range`.
- **`entailment="owl-rl"`** (default `infer="changed"`) — for genuine OWL constructs RDFS rewriting can't express, recomputed only when the graph actually changed since the last call. The right default for most uses: correct and cheap on repeated queries against unchanged data.
- **`infer="cached"`** — when you'd rather control refresh timing yourself (e.g. reason once, then run a batch of queries you know shouldn't see a change mid-batch) than have every call pay for a staleness check.
- **`infer="always"`** — when you don't trust the cache for a particular call (e.g. debugging, or data changed some way you're not sure the graph's own mutation tracking saw).

## Further Reading

1. **[Getting Started](01-getting-started.ipynb)** — install, first parse, first query, first validate.
2. **[Graphs](02-graphs.ipynb)** — `TripleTerm`/`DirLangString` semantics, Turtle 1.2 reification syntax.
   - 2.b **[Inferencing](02b-graphs-inferencing.ipynb)** — RDFS/OWL-RL reasoning via `owlrl`, including `StarLayerGraph.infer()`.
3. **SPARQL**
   - 3.a **[SPARQL rules (pending)](03a-sparql-rules-pending.md)** — SPARQL-RL (SRL), a separate Datalog-style rules language, deliberately out of scope for this project.
   - 3.b **SPARQL inferencing** — this guide.
5. **Other**
   - 5.d **[Canonical hashing and graph comparison](05d-canonical-hashing.ipynb)** — RDFC-1.0 canonicalization/hashing and graph isomorphism.